# BioCLIP prompt formats on the Rare Species dataset

Replaces the three 2024 notebooks `rarespecies-common.ipynb`, `rarespecies-common-taxi.ipynb`
and `rarespecies-common-taxi-nointro.ipynb` (archived verbatim). The dataset is
[`imageomics/rare-species`](https://huggingface.co/datasets/imageomics/rare-species):
400 IUCN-listed species, 11,983 Encyclopedia of Life images, held out of BioCLIP's training data.

**Historical recorded outputs (2024, not reproduced here):**

| prompt | labels | top-1 | source |
|---|---|---|---|
| `a photo of {common}` | 398 | 0.2944 (3528 / 11983) | `rarespecies-common.ipynb` |
| `a photo of {taxonomy} with common name {common}` | 400 | 0.3717 | `rarespecies-common-taxi.ipynb` |
| `{taxonomy} with common name {common}` | 400 | 0.3739 | `rarespecies-common-taxi-nointro.ipynb` |

The presentation (slide 19) also shows bars for *scientific* (0.313), *taxonomic* (0.327) and
*common + scientific* (0.36); no notebook or artifact for those three exists in the repository.

### Why 398 vs 400 matters

Only 398 of the 400 species have a distinct common name: *Acropora acuminata* and *A. millepora*
are both "staghorn coral", and *Tylototriton shanjing* and *T. verrucosus* are both "Yunnan Newt".
The 2024 common-name notebook built its label list from unique common names, so those four
species were silently merged into two labels: a 398-way task, scored leniently for the merged pairs.
The taxonomic notebooks had 400 distinct labels, so the three historical numbers are not
measured on the same task.

Here species identity is always the 400 scientific names. Shared prompt text produces identical
text embeddings and therefore exact score ties, which are resolved by an explicit policy:

* `first`: argmax picks the lowest species index, so one species of each pair can never be predicted.
* `any`: the image counts as correct if its true species ties for the best score (equivalent to the
  2024 merged scoring).

Both are reported for every prompt format, so comparisons use one task (400 species) with a
stated tie rule. The 2024 notebooks also built label lists with `range(len(dataset) - 2)`; the
two skipped rows belonged to species already seen, so the omission had no effect, and it is not
carried over.


> **Smoke test vs. real run.** When the dataset is not present under `BIODIV_DATA_ROOT`
> (or `BIODIV_SMOKE_TEST=1` is set) this notebook runs on a tiny synthetic fixture with a
> weight-free stand-in encoder. That exercises the code path only; the numbers it prints are
> **not** results. A real run needs the dataset and downloads the pinned BioCLIP checkpoint
> (~600 MB per model) from the Hugging Face Hub.

In [ ]:
import json, os, sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path[:0] = [str(REPO / "src"), str(REPO)]
os.environ.setdefault("BIODIV_DATA_ROOT", str(REPO / "data"))      # where inat_comp/<year>/ lives
os.environ.setdefault("BIODIV_OUTPUT_DIR", str(REPO / "outputs"))

from biodiv.paths import get_paths
paths = get_paths()  # BIODIV_DATA_ROOT / BIODIV_CACHE_DIR / BIODIV_OUTPUT_DIR
print("data_root =", paths.data_root.resolve())

In [ ]:
SMOKE = os.environ.get("BIODIV_SMOKE_TEST", "auto")
RARE_JSONL = os.environ.get("BIODIV_RARE_SPECIES_JSONL")  # optional local export; otherwise the Hub is used
if SMOKE == "auto":
    SMOKE = "0" if (RARE_JSONL or os.environ.get("BIODIV_USE_HUB") == "1") else "1"
SMOKE = SMOKE == "1"
LIMIT = int(os.environ.get("BIODIV_LIMIT", "0")) or None
print("mode:", "SMOKE TEST (synthetic fixture, no weights)" if SMOKE else "REAL DATA")
print("Set BIODIV_USE_HUB=1 (downloads ~2 GB via the `datasets` library) or BIODIV_RARE_SPECIES_JSONL for a real run.")

In [ ]:
from biodiv.clip_eval import evaluate_zero_shot, duplicate_prompt_groups
from biodiv.species_data import rare_species_from_hub, rare_species_from_jsonl, RARE_SPECIES_REVISION

if SMOKE:
    from biodiv.fixtures import load_fixture_rare_species
    records, table = load_fixture_rare_species()
elif RARE_JSONL:
    records, table = rare_species_from_jsonl(RARE_JSONL)
else:
    records, table = rare_species_from_hub(cache_dir=str(paths.cache_dir), revision=RARE_SPECIES_REVISION)
if LIMIT:
    records = records[:LIMIT]

common_prompts = table.prompts("common")
print(f"{len(table)} species | {len(records)} images | {len(set(common_prompts))} unique common names")
for text, ids in duplicate_prompt_groups(common_prompts).items():
    print(f"  shared common name {text!r}: {[table[i].scientific for i in ids]}")

In [ ]:
from biodiv.encoders import KNOWN_CHECKPOINTS, OpenClipEncoder

if SMOKE:
    from biodiv.fixtures import fixture_encoder
    encoder = fixture_encoder()
else:
    encoder = OpenClipEncoder("imageomics/bioclip", revision=KNOWN_CHECKPOINTS["imageomics/bioclip"], cache_dir=str(paths.cache_dir))
print(encoder.describe())

In [ ]:
FORMATS = [  # (style, prefix) -- the six bars of slide 19, in the same order
    ("common", "a photo of "),
    ("scientific", "a photo of "),
    ("taxonomic", "a photo of "),
    ("scientific_common", "a photo of "),
    ("taxonomic_common", "a photo of "),
    ("taxonomic_common", ""),
]
rows = []
for style, prefix in FORMATS:
    r = evaluate_zero_shot(encoder, records, table, style, prefix, batch_size=32, num_workers=0)
    rows.append(r)
    print(f"{style:18s} prefix={prefix!r:14s} unique prompts {r['n_unique_prompts']:4d} | "
          f"top-1 first {r['top1_accuracy_first']:.4f} | top-1 any {r['top1_accuracy_any']:.4f} | top-5 {r['top5_accuracy']:.4f} | "
          f"n={r['n_evaluated']} skipped={r['n_skipped']}")

In [ ]:
out = paths.output_dir / ("smoke_bioclip_rare_species.json" if SMOKE else "bioclip_rare_species.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"smoke_test": SMOKE, "limit": LIMIT, "results": rows}, indent=2))
print("wrote", out)

## Notes

* The BioCLIP paper reports 38.09% zero-shot top-1 on Rare Species with taxonomic + common prompts
  (dataset card); the 2024 value of 37.17% / 37.39% is in the same range but was measured with the
  code described above, so differences should not be over-interpreted.
* Images are read with Pillow; one image in the dataset exceeds Pillow's decompression-bomb
  warning threshold (the 2024 logs show the warning). It is still decoded; a failure would appear
  under `n_skipped` instead of shifting labels.
